# **Language Ablation**

**Question:** does adding training data from *related* languages improve Tagalog grapheme-to-phoneme (G2P) conversion — and does *how related* they are matter?

We train three Transformer G2P models that are identical in every way (architecture, hyperparameters, seed) **except for their training data**:

| Set | Training languages |
|---|---|
| `ph` | Tagalog + 7 other Philippine languages |
| `austro` | Tagalog + non-Philippine Austronesian languages (Indonesian, Malay, Acehnese, …) |
| `all` | Tagalog + both groups combined |

All three are evaluated on the **same held-out Tagalog test set** using phoneme error rate (PER) and word error rate (WER). If genetic proximity drives cross-lingual transfer, the Philippine mix should beat the more distantly related Austronesian mix.

## Setup

This notebook runs on Colab: clone a fresh copy of the repo, move into `notebooks/` (all paths below are relative to it), and import the project modules from `src/` — data prep (`prep_data`), beam-search decoding (`decode`), the Transformer/training utilities (`train`), and temperature-based language sampling (`temperature`).

In [21]:
# %cd /content
# !rm -rf /content/Hunger

# import os
# REPO_DIR = "/content/Hunger"
# if not os.path.exists(REPO_DIR):
#     !git clone https://github.com/OutForMilks/Hunger {REPO_DIR}
# %cd {REPO_DIR}
# !pwd

In [22]:
# %cd notebooks

In [23]:
import csv
import pandas as pd
import os
from pathlib import Path
import glob

import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler

import sys
parent_dir = str(Path.cwd().parent)
sys.path.append(parent_dir)

from src.prep_data import *
from src.decode import *
from src.train import *
from src.temperature import *
from src.evaluation import *

## Cleaning & Organizing

Load the raw WikiPron TSVs (one file per language) into dataframes. Each row is a `(text, phonetic)` pair — a spelled word and its space-separated IPA transcription. Every row gets tagged with its `language` (taken from the filename) and its ablation `group` (`philippine` or `austronesian`). Duplicate and null rows are dropped later, once the Tagalog train split has been mixed in.

In [24]:
input_dir_ph = Path("./wikipron/filipino")
input_dir_austro = Path("./wikipron/austronesian")

# The two ablation groups: Philippine languages vs. other (non-Philippine) Austronesian languages.
# "west makian" has a space, but the file on disk is west_makian_*.tsv, so convert_split
# skips it later ("[skip] ... not found") -- the austro set effectively has 6 aux languages.
ph_languages = ["bikolano", "cebuano", "hiligaynon", "ilocano", "kapampangan", "pangasinan", "tagalog", "waray"]
austro_languages = ["acehnese", "balinese", "iban", "indonesian", "makasar", "malay", "west makian"]
# tagalog is already in ph_languages, so the "all" set lists it twice -- convert_split
# writes its training pairs twice (17220 -> 34440 in the training log), i.e. tagalog
# is upweighted ~2x in the "all" mix before temperature sampling.
all_languages = ph_languages + austro_languages + ["tagalog"]
col_names = ["text", "phonetic", "language", "group"]

In [25]:
ph_files = glob.glob("../wikipron/filipino/*.tsv")
austro_files = glob.glob("../wikipron/austronesian/*.tsv")

In [26]:
temp_ph = []
temp_austro = []

# One TSV per language: read it and tag every row with its language
# (from the filename stem) and its ablation group.
for filename in ph_files:
    temp_df = pd.read_csv(filename, sep='\t', names=col_names, header=None)
    temp_df["language"] = Path(filename).stem
    temp_df["group"] = "philippine"
    temp_ph.append(temp_df)

for filename in austro_files:
    temp_df = pd.read_csv(filename, sep='\t', names=col_names, header=None)
    temp_df["language"] = Path(filename).stem
    temp_df["group"] = "austronesian"
    temp_austro.append(temp_df)

ph_df = pd.concat(temp_ph, ignore_index=True)
austro_df = pd.concat(temp_austro, ignore_index=True)
all_df = pd.concat(temp_austro + temp_ph, ignore_index=True)

In [27]:
print(ph_df.head())
print("\n")
print(austro_df.head())

print("\n")
print(ph_df["language"].unique())
print("\n")
print(austro_df["language"].unique())
print("\n")
print(all_df["language"].unique())

      text       phonetic  language       group
0    Abril    ʔ a b ɾ i l  bikolano  philippine
1   Agosto  ʔ a ɡ o s t o  bikolano  philippine
2  Aguilar  ʔ a ɡ i l a ɾ  bikolano  philippine
3    Albay    ʔ a l b a j  bikolano  philippine
4   Alzaga  ʔ a l s a ɡ a  bikolano  philippine


      text      phonetic  language         group
0     Acèh       a c ɛ h  acehnese  austronesian
1  Aleuhat  a l ɯ h a t̠  acehnese  austronesian
2    abang       a b a ŋ  acehnese  austronesian
3     adèe      a d ɛ ə̯  acehnese  austronesian
4    aleue      a l ɯ ə̯  acehnese  austronesian


<StringArray>
[   'bikolano',     'cebuano',  'hiligaynon',     'ilocano', 'kapampangan',
  'pangasinan',       'waray']
Length: 7, dtype: str


<StringArray>
['acehnese', 'balinese', 'iban', 'indonesian', 'makasar', 'malay',
 'west_makian']
Length: 7, dtype: str


<StringArray>
[   'acehnese',    'balinese',        'iban',  'indonesian',     'makasar',
       'malay', 'west_makian',    'bikolano',     'cebuano

## Splitting

Tagalog is the *target* language, so it's the only one that gets a proper **70/15/15 train/dev/test split**. Only its train portion is mixed into the three training sets — dev and test stay held out, so every model is scored on Tagalog words none of them ever saw.

The auxiliary languages are used in full (no held-out portion needed — they only serve as extra training signal). Each language is written to its own `{lang}_train.tsv`, and `convert_split` then assembles each ablation set into the parallel `train.src` / `train.tgt` / `train.lang` files the training loop reads.

In [28]:
input_dir_tgl = Path("../wikipron/tagalog.tsv")

df_tgl = pd.read_csv(input_dir_tgl, sep='\t', names=col_names, header=None)
df_tgl["language"] = "tagalog"
df_tgl["group"] = "philippine"

In [29]:
from sklearn.model_selection import train_test_split

# 70/15/15 split: peel off 15% for test, then 17.65% of the remaining 85% (~15% overall) for dev
train_dev_tgl, test_tgl = train_test_split(df_tgl, test_size=0.15, random_state=42)
train_tgl, dev_tgl = train_test_split(train_dev_tgl, test_size=0.1765, random_state=42)

ph_aux_df = ph_df.copy()
austro_aux_df = austro_df.copy()

# Each training mix = the tagalog train split + its auxiliary language group
tagalog_only_df = train_tgl.copy()
ph_df = pd.concat([train_tgl, ph_aux_df], ignore_index=True)
austro_df = pd.concat([train_tgl, austro_aux_df], ignore_index=True)
all_df = pd.concat([train_tgl, ph_aux_df, austro_aux_df], ignore_index=True)

# Clean only now, so duplicates between the tagalog split and the aux data are caught too
for d in [tagalog_only_df, ph_df, austro_df, all_df]:
    d.drop_duplicates(subset=["text", "phonetic"], inplace=True)
    d.dropna(inplace=True)

In [30]:
# ph_df.to_csv("ablation/input/ph_train.tsv", sep="\t", index=False, header=False)
# austro_df.to_csv("ablation/input/austro_train.tsv", sep="\t", index=False, header=False)
# all_df.to_csv("ablation/input/all_train.tsv", sep="\t", index=False, header=False)
# dev_tgl.to_csv("ablation/input/tgl_dev.tsv", sep="\t", index=False, header=False)
# test_tgl.to_csv("ablation/input/tgl_test.tsv", sep="\t", index=False, header=False)

In [31]:
input_dir = Path("ablation/input")
input_dir.mkdir(parents=True, exist_ok=True)

output_dir = Path("ablation/splits")
output_dir.mkdir(parents=True, exist_ok=True)

# Pool all training data, dedupe, then write one {lang}_train.tsv per language
# so convert_split can assemble any subset of languages into a training set.
combined_df = pd.concat(
    [ph_aux_df, austro_aux_df, train_tgl], ignore_index=True
).drop_duplicates(subset=["text", "phonetic"])

for lang, group_df in combined_df.groupby("language"):
    group_df[["text", "phonetic"]].to_csv(
        input_dir / f"{lang}_train.tsv", sep="\t", index=False, header=False
    )
    print(f"{lang}: {len(group_df)} pairs")

# The held-out tagalog dev/test splits -- every model is evaluated on these
dev_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_dev.tsv", sep="\t", index=False, header=False)
test_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_test.tsv", sep="\t", index=False, header=False)


acehnese: 265 pairs
balinese: 295 pairs
bikolano: 5432 pairs
cebuano: 3031 pairs
hiligaynon: 165 pairs
iban: 552 pairs
ilocano: 827 pairs
indonesian: 18199 pairs
kapampangan: 900 pairs
makasar: 766 pairs
malay: 4323 pairs
pangasinan: 158 pairs
tagalog: 17220 pairs
waray: 186 pairs
west_makian: 740 pairs


In [32]:
# Assemble the three ablation training sets: each becomes parallel
# train.src (graphemes) / train.tgt (phonemes) / train.lang (language tag) files.
sets = {
    "ph": ph_languages,
    "austro": austro_languages,
    "all": all_languages,
}

for set_name, langs in sets.items():
    out_dir = f"ablation/splits/{set_name}"
    print(f"Set: {set_name}")
    convert_split(input_dir, out_dir, langs, "train")

# Dev/test are tagalog-only: shared across all three models
for split in ["dev", "test"]:
    out_dir = f"ablation/splits/tgl_{split}"
    print(f"Split: {split}")
    convert_split(input_dir, out_dir, ["tagalog"], split)

Set: ph
e:\College\Year 3\Term 3\DEEPLRN\Hunger\notebooks
  bikolano train: 5432 pairs
  cebuano train: 3031 pairs
  hiligaynon train: 165 pairs
  ilocano train: 827 pairs
  kapampangan train: 900 pairs
  pangasinan train: 158 pairs
  tagalog train: 17220 pairs
  waray train: 186 pairs
  -> wrote 27919 lines to train.{src,tgt,lang}
Set: austro
e:\College\Year 3\Term 3\DEEPLRN\Hunger\notebooks
  acehnese train: 265 pairs
  balinese train: 295 pairs
  iban train: 552 pairs
  indonesian train: 18199 pairs
  makasar train: 766 pairs
  malay train: 4323 pairs
  [skip] ablation\input\west makian_train.tsv not found
  -> wrote 24400 lines to train.{src,tgt,lang}
Set: all
e:\College\Year 3\Term 3\DEEPLRN\Hunger\notebooks
  bikolano train: 5432 pairs
  cebuano train: 3031 pairs
  hiligaynon train: 165 pairs
  ilocano train: 827 pairs
  kapampangan train: 900 pairs
  pangasinan train: 158 pairs
  tagalog train: 17220 pairs
  waray train: 186 pairs
  acehnese train: 265 pairs
  balinese train: 29

## Train

One model per ablation set, all sharing the exact same configuration and seed — the training data is the only variable, so any difference in test scores is attributable to the language mix.

The model is a standard encoder–decoder Transformer (6 layers, `d_model=512`, 8 heads, FF 2048) trained with the classic *Attention Is All You Need* recipe: Adam + Noam LR warmup, label smoothing 0.1, dropout 0.1. Each run is 1000 steps at batch size 256.

Because the language sizes are wildly imbalanced (158 Pangasinan pairs vs. ~18k Indonesian), batches are drawn with **temperature sampling** (T=5): the natural language distribution `p` is flattened into `q`, so low-resource languages are seen far more often than their raw share — the per-language `p → q` tables are printed at the start of each run.

In [33]:
DATA = Path("ablation/splits")
OUTPUT = Path("./models")
#seed is a var
STEPS = 1000
SAVE_STEP = 1000

# Transformer-base-style architecture (Vaswani et al. 2017)
BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

# Regularization + Noam schedule warmup steps
DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

# if len(xm.get_xla_supported_devices()) > 0:
#     DEVICE = "xla"
if torch.cuda.is_available():
    DEVICE = "cuda"
    # TF32 matmuls on Ampere+ tensor cores: big speedup, negligible precision loss
    torch.set_float32_matmul_precision("high")
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

# Saved into every checkpoint so a .pt file is self-describing
CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 100

cuda


In [34]:
# sets = ["ph", "austro", "all"]
# seed = 42

# if xla:

#     import torch_xla.distributed.parallel_loader as pl
#     device = xm.xla_device()
#     torch.manual_seed(seed)  # xm also seeds RNG on optimizer_step
# else:
#     device = torch.device(DEVICE)
#     torch.manual_seed(seed)

# # bf16 autocast on CUDA: tensor-core matmuls + roughly halved activation
# # memory; unlike fp16 it needs no GradScaler (same exponent range as fp32).
# use_amp = DEVICE == "cuda" and torch.cuda.is_bf16_supported()

# # Train one model per ablation set; everything below is identical across sets
# # except data_path/output_path.
# for set_name in sets:
#     print(f"\nTraining set: {set_name}")
#     DATA = Path(f"ablation/splits/{set_name}")
#     OUTPUT = Path(f"./models/{set_name}")
#     os.makedirs(OUTPUT, exist_ok=True)

#     config_1 = dict(CONFIG)
#     config_1["seed"] = seed
#     config_1["data_path"] = str(DATA)
#     config_1["output_path"] = str(OUTPUT)

#     # Re-seed per run so each model trains from the same init regardless of set order
#     torch.manual_seed(seed)
#     device = torch.device(DEVICE)

#     # Vocab is built from this set's training data only (each set has its own
#     # grapheme/phoneme inventory), then saved alongside the checkpoints.
#     src_vocab, tgt_vocab = build_vocabs(
#         os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
#     save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

#     train_ds = G2PDataset(os.path.join(DATA, "train.src"),
#                             os.path.join(DATA, "train.tgt"),
#                             src_vocab, tgt_vocab)
#     # Fixed max lengths keep every batch the same shape -- required on XLA.
#     # On GPU, pad each batch only to its own max instead: mean word length is
#     # ~8 graphemes vs. a global max of ~48, so this skips most padding compute.
#     max_src, max_tgt = compute_max_lengths(train_ds)
#     print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
#             f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

#     # Temperature sampling (T=5) flattens the language imbalance: low-resource
#     # languages get sampled far above their natural frequency.
#     weights = temperature_sampling_weights(
#         os.path.join(DATA, "train.lang"), temperature=5.0
#     )
#     sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

#     collate_fn = (make_fixed_collate(max_src, max_tgt) if xla
#                   else make_dynamic_collate())
#     loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
#                         collate_fn=collate_fn,
#                         drop_last=True)
#     if xla:
#         loader = pl.MpDeviceLoader(loader, device)  # preloads batches onto the TPU

#     model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
#                         n_heads=NUM_HEADS, d_ff=FF_SIZE,
#                         n_layers=N_LAYERS, dropout=DROPOUT,
#                         pad_id=PAD_ID).to(device)
#     # lr=0 because NoamLR fully controls the learning rate (warmup then decay)
#     opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
#     sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
#     criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

#     model.train()
#     data_iter = infinite_loader(loader)
#     for step in range(1, STEPS + 1):
#         batch = next(data_iter)
#         if xla:
#             src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
#         else:
#             src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

#         # Teacher forcing: tgt_in is the shifted target, tgt_out is what we score against
#         with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
#             logits = model(src, tgt_in, src_pad, tgt_pad)
#             loss = criterion(logits, tgt_out)
#         opt.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

#         if step % log_step == 0:
#             # .item() forces a host sync; only do it at log intervals on TPU.
#             print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
#                     f"lr {opt.param_groups[0]['lr']:.2e}")

#         if step % SAVE_STEP == 0 or step == STEPS:
#             # Checkpoint bundles weights + config + both vocabs, so evaluation
#             # can rebuild the model from the .pt file alone.
#             ckpt = os.path.join(OUTPUT, f"ckpt_{step}.pt")
#             payload = {"model": model.state_dict(),
#                         "config": config_1,
#                         "src_vocab": src_vocab.to_dict(),
#                         "tgt_vocab": tgt_vocab.to_dict()}
#             if xla:
#                 xm.save(payload, ckpt)   # moves tensors to CPU; master-only
#             else:
#                 torch.save(payload, ckpt)
#             print(f"  saved {ckpt}")


In [35]:
# for saving the .pt model stuff from colab if needed
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
# !mkdir -p /content/drive/MyDrive/Hunger_checkpoints
# !cp -r /content/Hunger/notebooks/models/* /content/drive/MyDrive/Hunger_checkpoints/

## Evaluation

Each model decodes the held-out Tagalog test set (4,245 words) with beam search (`beam=5`) and is scored on:

- **PER** (phoneme error rate) — edit distance between predicted and reference phoneme sequences, normalized by reference length; and
- **WER** (word error rate) — fraction of words whose transcription isn't an exact match.

Lower is better for both. Since the models differ only in training mix, these two numbers *are* the ablation result.

In [36]:
# from src.evaluation import *

# device = torch.device(DEVICE)
# print(device)

# # Score every ablation model on the same held-out tagalog test set
# results = {}
# for set_name in ["ph", "austro", "all"]:
#     print(f"\nSet: {set_name}")
#     ckpt_path = f"./models/{set_name}/ckpt_{STEPS}.pt"

#     per, wer, refs, hyps = evaluate_checkpoint(
#         model_paths=[ckpt_path],
#         src_path="ablation/splits/tgl_test/test.src",
#         tgt_path="ablation/splits/tgl_test/test.tgt",
#         device=device,
#         beam=5,
#     )
#     results[set_name] = {"PER": per, "WER": wer}
#     print(f"{set_name}: PER={per:.4f}  WER={wer:.4f}")

# print("\nSummary:")
# for name, r in results.items():
#     print(f"{name}  PER={r['PER']:.4f}  WER={r['WER']:.4f}")

### Results & takeaways

| Training set | PER ↓ | WER ↓ |
|---|---|---|
| `ph` (Tagalog + Philippine) | **0.2004** | **0.5559** |
| `austro` (Tagalog + other Austronesian) | 0.4438 | 0.8912 |
| `all` (everything) | 0.2222 | 0.6040 |

**Genetic proximity clearly matters.** The Philippine mix (`ph`) is the best model on both metrics, and by a wide margin: swapping the Philippine auxiliary languages for the more distantly related Austronesian ones (`austro`) roughly *doubles* PER (0.20 → 0.44) and pushes WER close to 0.90 — i.e. that model gets almost nine out of ten Tagalog words wrong. So the extra data only helps when it is closely related to the target; distant Austronesian data is nearly useless as transfer signal here.

**More data is not better data.** The `all` set has strictly more training pairs than `ph` (it *is* `ph` plus the Austronesian languages), yet it lands *between* the two — worse than `ph` on both metrics. Pouring in the distant languages dilutes the useful Philippine signal rather than adding to it, confirming the hypothesis in the intro: for cross-lingual G2P transfer into Tagalog, *how related* the auxiliary languages are outweighs *how many* pairs they contribute. We therefore carry the `ph` mix forward as the base for the Spanish and temperature-tuning experiments below.

## Adding Spanish

The ablation above only considered *Austronesian* auxiliary languages. But Tagalog also carries centuries of Spanish influence: a large share of its vocabulary is borrowed from Spanish (numbers, days, months, religious and administrative terms), and those loanwords tend to keep Spanish-like spelling-to-sound patterns. Spanish is not genetically related to Tagalog at all, but it may still be a useful *contact* source — its grapheme→phoneme regularities could transfer to exactly the Spanish-derived slice of Tagalog vocabulary.

So we take the winning `ph` mix and add a Castilian Spanish set on top of it (`ph_spanish`). Spanish also brings a very large, phonetically consistent corpus (~132k pairs), which is why the next section pairs it with a proper temperature sweep — without careful reweighting a set that big would swamp every Philippine language in the batch.

### Cleaning

Load the pre-filtered Castilian Spanish TSV, tag every row with `language`/`group` = `spanish`, drop duplicate and null `(text, phonetic)` pairs, and write it out as `spanish_train.tsv` in the same `ablation/input/` layout as the other per-language files. Once it lives there, `convert_split` can fold Spanish into any training set exactly like a Philippine language — here we assemble the `ph_spanish` set (the Philippine mix plus Spanish, ~160k pairs total).

In [37]:
col_names = ["text", "phonetic"]

df_spa = pd.read_csv("../wikipron/castillian_spanish_filtered.tsv", sep='\t', names=col_names, header=None)
df_spa["language"] = "spanish"
df_spa["group"] = "spanish"

df_spa = df_spa.drop_duplicates(subset=["text", "phonetic"]).dropna()
df_spa[["text", "phonetic"]].to_csv("ablation/input/spanish_train.tsv", sep="\t", index=False, header=False)

In [38]:
sets = {
    "ph_spanish": ph_languages + ["spanish"],
}

input_dir = Path("ablation/input")

for set_name, langs in sets.items():
    out_dir = f"ablation/splits/{set_name}"
    print(f"Set: {set_name}")
    convert_split(input_dir, out_dir, langs, "train")

Set: ph_spanish
e:\College\Year 3\Term 3\DEEPLRN\Hunger\notebooks
  bikolano train: 5432 pairs
  cebuano train: 3031 pairs
  hiligaynon train: 165 pairs
  ilocano train: 827 pairs
  kapampangan train: 900 pairs
  pangasinan train: 158 pairs
  tagalog train: 17220 pairs
  waray train: 186 pairs
  spanish train: 132321 pairs
  -> wrote 160240 lines to train.{src,tgt,lang}


## Temperature Tuning - Spanish

Since the Philippine language set performed the best on the Tagalog test set, we use it plus the Castilian Spanish set (`ph_spanish`) as the base for tuning the sampling temperature.

Temperature `T` controls how aggressively the language imbalance is flattened before batching: higher `T` upweights the low-resource Philippine languages more, at the cost of seeing the huge Spanish and Indonesian sets proportionally less. With Spanish's ~132k pairs now dominating the pool, the right `T` is no longer obvious, so we sweep **T ∈ {2, 3, 4, 5, 6}**.

Each temperature trains its own model from the same seed for **10,000 steps** (10× the earlier ablation runs) and is scored on the held-out Tagalog **dev** set — the test set stays untouched so it can serve as a clean final evaluation. Checkpoints are cached per temperature (`ckpt_temp_{T}.pt`), so re-running the sweep skips any temperature already trained.

In [ ]:
#seed is a var
STEPS = 10000
SAVE_STEP = 10000

# Transformer-base-style architecture (Vaswani et al. 2017)
BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

# Regularization + Noam schedule warmup steps
DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

# if len(xm.get_xla_supported_devices()) > 0:
#     DEVICE = "xla"
if torch.cuda.is_available():
    DEVICE = "cuda"
    # TF32 matmuls on Ampere+ tensor cores: big speedup, negligible precision loss
    torch.set_float32_matmul_precision("high")
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

# Saved into every checkpoint so a .pt file is self-describing
CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 1000

cuda


In [40]:
seed = 42
temps = [2, 3, 4, 5, 6]
temp_results = {}


device = torch.device(DEVICE)
torch.manual_seed(seed)

# bf16 autocast on CUDA: tensor-core matmuls + roughly halved activation
# memory; unlike fp16 it needs no GradScaler (same exponent range as fp32).
use_amp = DEVICE == "cuda" and torch.cuda.is_bf16_supported()

# Train one model per ablation set; everything below is identical across sets
# except data_path/output_path.
for temperature in temps:
    print(f"\nTemperature: {temperature}")
    DATA = Path(f"ablation/splits/ph_spanish")
    OUTPUT = Path(f"./models/ph_spanish")
    os.makedirs(OUTPUT, exist_ok=True)
    ckpt_path = os.path.join(OUTPUT, f"ckpt_temp_{temperature}.pt")

    if not os.path.exists(ckpt_path):
        config_1 = dict(CONFIG)
        config_1["seed"] = seed
        config_1["data_path"] = str(DATA)
        config_1["output_path"] = str(OUTPUT)
        config_1["temperature"] = temperature

        # Re-seed per run so each model trains from the same init regardless of set order
        torch.manual_seed(seed)
        device = torch.device(DEVICE)

        # Vocab is built from this set's training data only (each set has its own
        # grapheme/phoneme inventory), then saved alongside the checkpoints.
        src_vocab, tgt_vocab = build_vocabs(
            os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
        save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

        train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                                os.path.join(DATA, "train.tgt"),
                                src_vocab, tgt_vocab)
        # Fixed max lengths keep every batch the same shape -- required on XLA.
        # On GPU, pad each batch only to its own max instead: mean word length is
        # ~8 graphemes vs. a global max of ~48, so this skips most padding compute.
        max_src, max_tgt = compute_max_lengths(train_ds)
        print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
                f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

        # Temperature sampling (T=5) flattens the language imbalance: low-resource
        # languages get sampled far above their natural frequency.
        weights = temperature_sampling_weights(
            os.path.join(DATA, "train.lang"), temperature=temperature
        )
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

        collate_fn = (make_fixed_collate(max_src, max_tgt) if xla
                      else make_dynamic_collate())
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                            collate_fn=collate_fn,
                            drop_last=True)

        model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                            n_heads=NUM_HEADS, d_ff=FF_SIZE,
                            n_layers=N_LAYERS, dropout=DROPOUT,
                            pad_id=PAD_ID).to(device)
        # lr=0 because NoamLR fully controls the learning rate (warmup then decay)
        opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
        sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
        criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

        model.train()
        data_iter = infinite_loader(loader)
        for step in range(1, STEPS + 1):
            batch = next(data_iter)
            if xla:
                src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
            else:
                src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

            # Teacher forcing: tgt_in is the shifted target, tgt_out is what we score against
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
                logits = model(src, tgt_in, src_pad, tgt_pad)
                loss = criterion(logits, tgt_out)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

            if step % log_step == 0:
                # .item() forces a host sync; only do it at log intervals on TPU.
                print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                        f"lr {opt.param_groups[0]['lr']:.2e}")

            if step % SAVE_STEP == 0 or step == STEPS:
                # Checkpoint bundles weights + config + both vocabs, so evaluation
                # can rebuild the model from the .pt file alone.
                payload = {"model": model.state_dict(),
                            "config": config_1,
                            "src_vocab": src_vocab.to_dict(),
                            "tgt_vocab": tgt_vocab.to_dict()}
                torch.save(payload, ckpt_path)
                print(f"  saved {ckpt_path}")

        del model, opt, sched, criterion, loader, train_ds
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    per, wer, _, _ = batch_evaluate_checkpoint(
        model_paths=[ckpt_path],
        src_path="ablation/splits/tgl_dev/dev.src",
        tgt_path="ablation/splits/tgl_dev/dev.tgt",
        device=device,
        batch_size=64,
    )
    temp_results[temperature] = {"PER": per, "WER": wer}
    print(f"T={temperature}: dev PER={per:.4f}  WER={wer:.4f}")



Temperature: 2
  decoded 64/4245
  decoded 128/4245
  decoded 192/4245
  decoded 256/4245
  decoded 320/4245
  decoded 384/4245
  decoded 448/4245
  decoded 512/4245
  decoded 576/4245
  decoded 640/4245
  decoded 704/4245
  decoded 768/4245
  decoded 832/4245
  decoded 896/4245
  decoded 960/4245
  decoded 1024/4245
  decoded 1088/4245
  decoded 1152/4245
  decoded 1216/4245
  decoded 1280/4245
  decoded 1344/4245
  decoded 1408/4245
  decoded 1472/4245
  decoded 1536/4245
  decoded 1600/4245
  decoded 1664/4245
  decoded 1728/4245
  decoded 1792/4245
  decoded 1856/4245
  decoded 1920/4245
  decoded 1984/4245
  decoded 2048/4245
  decoded 2112/4245
  decoded 2176/4245
  decoded 2240/4245
  decoded 2304/4245
  decoded 2368/4245
  decoded 2432/4245
  decoded 2496/4245
  decoded 2560/4245
  decoded 2624/4245
  decoded 2688/4245
  decoded 2752/4245
  decoded 2816/4245
  decoded 2880/4245
  decoded 2944/4245
  decoded 3008/4245
  decoded 3072/4245
  decoded 3136/4245
  decoded 3200/4245


In [41]:
print("\nTemperature summary:")
for t, r in sorted(temp_results.items(), key=lambda x: x[1]["PER"]):
    print(f"T={t}: PER={r['PER']:.4f}  WER={r['WER']:.4f}")

best_temp = min(temp_results, key=lambda t: temp_results[t]["PER"])
print(f"\nBest temperature: {best_temp}")


Temperature summary:
T=3: PER=0.0542  WER=0.2052
T=4: PER=0.0733  WER=0.2349
T=2: PER=0.0757  WER=0.2120
T=6: PER=0.0760  WER=0.2179
T=5: PER=0.0930  WER=0.2297

Best temperature: 3


### Temperature results & takeaways

| Temperature | dev PER ↓ | dev WER ↓ |
|---|---|---|
| **T=3** | **0.0542** | **0.2052** |
| T=4 | 0.0733 | 0.2349 |
| T=2 | 0.0757 | 0.2120 |
| T=6 | 0.0760 | 0.2179 |
| T=5 | 0.0930 | 0.2297 |

**T=3 is the winner**, best on both PER and WER. The relationship is not monotonic — PER does not simply fall or rise with `T` — but there is a clear sweet spot around T=3: too low (T=2) and the big Spanish/Indonesian sets dominate the batches, too high (T=5–6) and the model over-samples tiny, noisy low-resource languages. So we adopt **T=3** for `ph_spanish`.

Note these dev PER/WER numbers (~0.05 / ~0.21) are far lower than the ablation table above (~0.20 / ~0.56) and are **not directly comparable**: this sweep trains 10× longer (10,000 vs 1,000 steps), adds the large Spanish corpus, and reports on the *dev* split rather than *test*. The takeaway is the relative ranking across temperatures, not the absolute drop versus the earlier experiment.

## Adding US English

In [ ]:
col_names = ["text", "phonetic"]

df_eng = pd.read_csv("../wikipron/us_english_filtered.tsv", sep='\t', names=col_names, header=None)
df_eng["language"] = "english"
df_eng["group"] = "english"

df_eng = df_eng.drop_duplicates(subset=["text", "phonetic"]).dropna()
df_eng[["text", "phonetic"]].to_csv("ablation/input/english_train.tsv", sep="\t", index=False, header=False)

In [ ]:
sets = {
    "ph_english": ph_languages + ["english"],
}

input_dir = Path("ablation/input")

for set_name, langs in sets.items():
    out_dir = f"ablation/splits/{set_name}"
    print(f"Set: {set_name}")
    convert_split(input_dir, out_dir, langs, "train")

## Temperature Tuning - English

In [ ]:
seed = 42
temps = [2, 3, 4, 5, 6]
temp_results = {}


device = torch.device(DEVICE)
torch.manual_seed(seed)

# bf16 autocast on CUDA: tensor-core matmuls + roughly halved activation
# memory; unlike fp16 it needs no GradScaler (same exponent range as fp32).
use_amp = DEVICE == "cuda" and torch.cuda.is_bf16_supported()

# Train one model per ablation set; everything below is identical across sets
# except data_path/output_path.
for temperature in temps:
    print(f"\nTemperature: {temperature}")
    DATA = Path(f"ablation/splits/ph_english")
    OUTPUT = Path(f"./models/ph_english")
    os.makedirs(OUTPUT, exist_ok=True)
    ckpt_path = os.path.join(OUTPUT, f"ckpt_temp_{temperature}.pt")

    if not os.path.exists(ckpt_path):
        config_1 = dict(CONFIG)
        config_1["seed"] = seed
        config_1["data_path"] = str(DATA)
        config_1["output_path"] = str(OUTPUT)
        config_1["temperature"] = temperature

        # Re-seed per run so each model trains from the same init regardless of set order
        torch.manual_seed(seed)
        device = torch.device(DEVICE)

        # Vocab is built from this set's training data only (each set has its own
        # grapheme/phoneme inventory), then saved alongside the checkpoints.
        src_vocab, tgt_vocab = build_vocabs(
            os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
        save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

        train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                                os.path.join(DATA, "train.tgt"),
                                src_vocab, tgt_vocab)
        # Fixed max lengths keep every batch the same shape -- required on XLA.
        # On GPU, pad each batch only to its own max instead: mean word length is
        # ~8 graphemes vs. a global max of ~48, so this skips most padding compute.
        max_src, max_tgt = compute_max_lengths(train_ds)
        print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
                f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

        # Temperature sampling (T=5) flattens the language imbalance: low-resource
        # languages get sampled far above their natural frequency.
        weights = temperature_sampling_weights(
            os.path.join(DATA, "train.lang"), temperature=temperature
        )
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

        collate_fn = (make_fixed_collate(max_src, max_tgt) if xla
                      else make_dynamic_collate())
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                            collate_fn=collate_fn,
                            drop_last=True)

        model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                            n_heads=NUM_HEADS, d_ff=FF_SIZE,
                            n_layers=N_LAYERS, dropout=DROPOUT,
                            pad_id=PAD_ID).to(device)
        # lr=0 because NoamLR fully controls the learning rate (warmup then decay)
        opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
        sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
        criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

        model.train()
        data_iter = infinite_loader(loader)
        for step in range(1, STEPS + 1):
            batch = next(data_iter)
            if xla:
                src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
            else:
                src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

            # Teacher forcing: tgt_in is the shifted target, tgt_out is what we score against
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
                logits = model(src, tgt_in, src_pad, tgt_pad)
                loss = criterion(logits, tgt_out)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

            if step % log_step == 0:
                # .item() forces a host sync; only do it at log intervals on TPU.
                print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                        f"lr {opt.param_groups[0]['lr']:.2e}")

            if step % SAVE_STEP == 0 or step == STEPS:
                # Checkpoint bundles weights + config + both vocabs, so evaluation
                # can rebuild the model from the .pt file alone.
                payload = {"model": model.state_dict(),
                            "config": config_1,
                            "src_vocab": src_vocab.to_dict(),
                            "tgt_vocab": tgt_vocab.to_dict()}
                torch.save(payload, ckpt_path)
                print(f"  saved {ckpt_path}")

        del model, opt, sched, criterion, loader, train_ds
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    per, wer, _, _ = batch_evaluate_checkpoint(
        model_paths=[ckpt_path],
        src_path="ablation/splits/tgl_dev/dev.src",
        tgt_path="ablation/splits/tgl_dev/dev.tgt",
        device=device,
        batch_size=64,
    )
    temp_results[temperature] = {"PER": per, "WER": wer}
    print(f"T={temperature}: dev PER={per:.4f}  WER={wer:.4f}")
